In [1]:
# import libraries

#import libraries to conduct eda
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import os

# import for distance calculations
!pip install geopy
from geopy import distance
import math

# import scipy for statistical analysis in eda
#!pip install scipy
from scipy import stats

# import datetime to set timestamp on pdf report
from datetime import datetime, timedelta

print("All required libraries installed")


All required libraries installed


In [2]:
# data file names 

data_file_hurr = "../data/ibtracs.NA.list.v04r01.csv"

data_file_gas = "../data/aaa_fl_metros_wayback_2022_2025.csv"
#"../data/gas_coordinates.csv"

data_file_coast = "../data/Gulf_Coast_Coords.csv"

data_file_metro_loc = "../data/gas_coordinates.csv"

In [3]:
# import file loading functions
from file_load import *

In [4]:
# import data files

# import data files
hurr_file_chk = input('Is hurricane data file downloaded to data folder? (y/n) ')
if hurr_file_chk == 'y':
    hurr_data = df_import(data_file_hurr)
else:
    url = 'https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/ibtracs.NA.list.v04r01.csv'
    hurr_data = pd.read_csv(url)
    
hurr_data.name = 'hurr_data'

gas_data = df_import(data_file_gas)
gas_data.name = 'gas_data'

coast_pt_data = df_import(data_file_coast)

metro_loc_data = coord_import(data_file_metro_loc)

Is hurricane data file downloaded to data folder? (y/n)  y


C:\Users\eenoo\Downloads\CSCI_5502\Project\src\file_load.py:6: DtypeWarning: Columns (0: SEASON, 1: NUMBER, 2: BASIN, 3: LAT, 4: LON, 5: DIST2LAND, 6: USA_LAT, 7: USA_LON, 8: USA_WIND, 9: USA_PRES, 10: STORM_SPEED, 11: STORM_DIR) have mixed types. Specify dtype option on import or set low_memory=False.
  pri_data = pd.read_csv(datafile)


Successfully loaded ../data/ibtracs.NA.list.v04r01.csv
Successfully loaded ../data/aaa_fl_metros_wayback_2022_2025.csv
Successfully loaded ../data/Gulf_Coast_Coords.csv
Successfully loaded ../data/gas_coordinates.csv


In [5]:
# import data cleaning functions
from clean import *

In [6]:
# Clean hurricane data and generate data frames for cleaned hurricane data, list of hurricanes, and summary list of hurricanes with maximum storm category observed
# Generates data frames with ALL data and a set of filtered data frames for "recent" (2022-2025)
hurr_data_redux, hurr_list, hurr_sum_list, hurr_data_redux_rec, hurr_list_rec, hurr_sum_list_rec, hurr_data_redux_rec_hur = hurr_clean(hurr_data)

# Name output data frames
hurr_data_redux.name = 'hurr_data_redux'
hurr_list.name = 'hurr_list'
hurr_sum_list.name = 'hurr_sum_list'
hurr_data_redux_rec.name = 'hurr_data_redux_rec'
hurr_list_rec.name = 'hurr_list_rec'
hurr_sum_list_rec.name = 'hurr_sum_list_rec'
hurr_data_redux_rec_hur.name = 'hurr_data_redux_rec_hur'

Cleaning hurr_data
          Unit
SID           
SEASON    Year
NUMBER        
BASIN         
SUBBASIN      
(174, 1)
Data table columns reduced to 47
Hurricane data cleaning complete


In [7]:
# List metro areas of interest
# all FL cities selected to represent the perimeter

metro_all = ['Pensacola', 'Tallahassee', 'Tampa-St. Petersburg-Clearwater', 'Fort Myers-Cape Coral', 'Miami','West Palm Beach-Boca Raton','Melbourne-Titusville','Daytona Beach','Jacksonville']

# East coast of FL
metro_east = ['Miami','West Palm Beach-Boca Raton','Melbourne-Titusville','Daytona Beach','Jacksonville']

# West coast of FL (exl panhandle)
metro_west = ['Tampa-St. Petersburg-Clearwater', 'Fort Myers-Cape Coral']

# Panhandle of FL
metro_ph = ['Pensacola', 'Tallahassee']

# Gulf Coast of FL (West + Panhandle)
metro_gulf = ['Pensacola', 'Tallahassee', 'Tampa-St. Petersburg-Clearwater', 'Fort Myers-Cape Coral']


In [8]:
# Clean gas station data and generate data frame for cleaned gas station data
gas_data_redux = gas_clean(gas_data)
gas_data_redux.name = 'gas_data_redux'

Cleaning gas_data
Gas data cleaning complete


In [9]:
# List of hurricanes from 2022 on
hurr_can_list = hurr_sum_list_rec[hurr_sum_list_rec["MAX CAT"] > 0]
hurr_can_list

,SID,SEASON,NAME,MAX CAT,START DATE,END DATE,STORM DURATION
2247,2022179N08310,2022,BONNIE,3,2022-06-27 18:00:00,2022-07-11 00:00:00,13
2250,2022244N38313,2022,DANIELLE,1,2022-08-31 12:00:00,2022-09-15 18:00:00,15
2251,2022246N18301,2022,EARL,2,2022-09-02 18:00:00,2022-09-15 12:00:00,12
2252,2022257N16312,2022,FIONA,4,2022-09-14 06:00:00,2022-09-27 18:00:00,13
2254,2022266N12294,2022,IAN,5,2022-09-22 18:00:00,2022-10-01 06:00:00,8
2258,2022280N11294,2022,JULIA,1,2022-10-06 12:00:00,2022-10-10 12:00:00,4
2260,2022304N16287,2022,LISA,1,2022-10-30 18:00:00,2022-11-05 06:00:00,5
2261,2022304N34296,2022,MARTIN,1,2022-10-30 18:00:00,2022-11-04 18:00:00,5
2262,2022311N21293,2022,NICOLE,1,2022-11-06 12:00:00,2022-11-11 18:00:00,5
2267,2023193N37305,2023,DON,1,2023-07-11 12:00:00,2023-07-25 18:00:00,14


In [13]:
# Additional cleaning of hurricane data set - removing additional unnecessary columns 
col_to_remove1 = ['SEASON_y','NUMBER','BASIN','SUBBASIN','NAME_y','LAT','LON','TRACK_TYPE','IFLAG','USA_AGENCY','USA_ATCF_ID']
col_to_remove2 = ['USA_R34_NE','USA_R34_SE','USA_R34_SW','USA_R34_NW','USA_R50_NE','USA_R50_SE','USA_R50_SW','USA_R50_NW','USA_R64_NE','USA_R64_SE','USA_R64_SW','USA_R64_NW','USA_SEARAD_NE','USA_SEARAD_SE','USA_SEARAD_SW','USA_SEARAD_NW']
hurr_mod_data = hurr_data_redux_rec_hur.drop(col_to_remove1, axis = 1)
hurr_mod_data.drop(col_to_remove2, axis = 1, inplace = True)
# fix/rename columns
hurr_mod_data.rename(columns={'SEASON_x': 'SEASON', 'NAME_x': 'NAME'}, inplace=True)
hurr_mod_data.columns

Index(['SID', 'SEASON', 'NAME', 'MAX CAT', 'START DATE', 'END DATE',
       'STORM DURATION', 'ISO_TIME', 'NATURE', 'DIST2LAND', 'LANDFALL',
       'USA_LAT', 'USA_LON', 'USA_RECORD', 'USA_STATUS', 'USA_WIND',
       'USA_PRES', 'USA_SSHS', 'USA_POCI', 'USA_ROCI', 'USA_RMW', 'USA_EYE',
       'USA_GUST', 'USA_SEAHGT', 'STORM_SPEED', 'STORM_DIR', 'DATE'],
      dtype='str')

In [15]:
hurr_mod_data.head()

,SID,SEASON,NAME,MAX CAT,START DATE,END DATE,STORM DURATION,ISO_TIME,NATURE,DIST2LAND,...,USA_SSHS,USA_POCI,USA_ROCI,USA_RMW,USA_EYE,USA_GUST,USA_SEAHGT,STORM_SPEED,STORM_DIR,DATE
0,2022179N08310,2022,BONNIE,3,2022-06-27 18:00:00,2022-07-11,13,2022-06-27 18:00:00,DS,417,...,-3,1012.0,150.0,120.0,NaN,40,NaN,17.0,285.0,2022-06-27
1,2022179N08310,2022,BONNIE,3,2022-06-27 18:00:00,2022-07-11,13,2022-06-27 21:00:00,DS,394,...,-3,1012.0,150.0,120.0,NaN,,NaN,17.0,285.0,2022-06-27
2,2022179N08310,2022,BONNIE,3,2022-06-27 18:00:00,2022-07-11,13,2022-06-28 00:00:00,DS,379,...,-3,1012.0,150.0,120.0,NaN,45,12.0,19.0,280.0,2022-06-28
3,2022179N08310,2022,BONNIE,3,2022-06-27 18:00:00,2022-07-11,13,2022-06-28 03:00:00,DS,353,...,-3,1012.0,150.0,120.0,NaN,,12.0,20.0,280.0,2022-06-28
4,2022179N08310,2022,BONNIE,3,2022-06-27 18:00:00,2022-07-11,13,2022-06-28 06:00:00,DS,358,...,-3,1012.0,150.0,120.0,NaN,45,12.0,22.0,280.0,2022-06-28


In [14]:
gas_data_redux.head()

,metro,date,regular,mid,premium,diesel,year,month,date_key,date_time_key
0,Bradenton-Sarasota-Venice,2022-01-21,3.236000,3.625,3.915000,3.563000,2022,1,2022-01-21,2022-01-21 12:00:00
1,Bradenton-Sarasota-Venice,2022-01-22,3.254667,3.639,3.929667,3.587667,2022,1,2022-01-22,2022-01-22 12:00:00
2,Bradenton-Sarasota-Venice,2022-01-23,3.273333,3.653,3.944333,3.612333,2022,1,2022-01-23,2022-01-23 12:00:00
3,Bradenton-Sarasota-Venice,2022-01-24,3.292000,3.667,3.959000,3.637000,2022,1,2022-01-24,2022-01-24 12:00:00
4,Bradenton-Sarasota-Venice,2022-01-25,3.310667,3.681,3.973667,3.661667,2022,1,2022-01-25,2022-01-25 12:00:00
